In [8]:
# Import required libraries
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [9]:
# Example: Load your dataframe (replace with your actual data loading method)
data = {
    'description': ['Base', 'Scene A', 'Scene B', 'Scene C', 'Back to Base'],
    'timestamp': [0, 10, 20, 30, 40],
    'level': [0, 1, 2, 3, 0]
}
df = pd.DataFrame(data)
df

,description,timestamp,level
0,Base,0,0
1,Scene A,10,1
2,Scene B,20,2
3,Scene C,30,3
4,Back to Base,40,0


In [10]:
descriptions=[
    "Interview", "Base Scene", "Tour of the Palace","At the Pub","KwikFit Kult",
    "Base Scene","Pyramid Scheme","Pyramid Scheme - 2nd Sale",
    "Base Scene","Jerry's Dry Cleaners","Laundry Drop Off","Auntie Wants That Hat","Sacrificed for a Hat",
    "Base Scene", "Hatters", "Police Enquiry",
    "Base Scene","Pack Your Bags Meghan",
    "Base Scene"
    ]
timestamps=[
    "00:00", "01:50","03:36","04:39","05:04",  
    "5:50","06:26","06:59",
    "7:51","8:39","9:06","9:38","10:07",
    "10:18", "10:38","11:36",
    "12:27","14:06",
    "14:38"]
levels=[
    0, 1, 2, 3, 4,
1,2,3,
1,2,3,4,5,
1,2,3,
1,2,
1]

#convert list of strings of format "MM:SS" to list of integers in seconds
def convert_to_seconds(timestamps):
    seconds = []
    for t in timestamps:
        m, s = map(int, t.split(':'))
        total_seconds = m * 60 + s
        seconds.append(total_seconds)
    return seconds

timestamps = convert_to_seconds(timestamps)

data = {
    'description': descriptions,
    'timestamp': timestamps,
    'level': levels
}

df = pd.DataFrame(data)
df

,description,timestamp,level
0,Interview,0,0
1,Base Scene,110,1
2,Tour of the Palace,216,2
3,At the Pub,279,3
4,KwikFit Kult,304,4
5,Base Scene,350,1
6,Pyramid Scheme,386,2
7,Pyramid Scheme - 2nd Sale,419,3
8,Base Scene,471,1
9,Jerry's Dry Cleaners,519,2


In [11]:
len(descriptions), len(timestamps), len(levels)

(19, 19, 19)

In [17]:
# Plot straight lines for constant levels and arcs for transitions, colored by level (no markers)
import plotly.colors as pc
from matplotlib import cm
from matplotlib.colors import Normalize, to_hex

levels = df['level'].unique()
norm = Normalize(vmin=min(levels), vmax=max(levels))
cmap = cm.get_cmap('viridis')
def get_level_color(level):
    return to_hex(cmap(norm(level)))

def bezier_arc(x0, y0, x1, y1, arc_height=0.5, steps=30):
    xm = (x0 + x1) / 2
    ym = (y0 + y1) / 2 + arc_height * (1 if y1 > y0 else -1)
    t = np.linspace(0, 1, steps)
    x = (1-t)**2 * x0 + 2*(1-t)*t * xm + t**2 * x1
    y = (1-t)**2 * y0 + 2*(1-t)*t * ym + t**2 * y1
    return x, y

# Convert seconds to MM:SS for x-axis labels
def seconds_to_mmss(seconds):
    m = int(seconds // 60)
    s = int(seconds % 60)
    return f"{m:02d}:{s:02d}"

df['timestamp_mmss'] = [seconds_to_mmss(ts) for ts in df['timestamp']]

fig = go.Figure()
for i in range(len(df)-1):
    x0, y0 = df['timestamp'][i], df['level'][i]
    x1, y1 = df['timestamp'][i+1], df['level'][i+1]
    # Use MM:SS for x-axis
    x0_mmss, x1_mmss = df['timestamp_mmss'][i], df['timestamp_mmss'][i+1]
    color = get_level_color(y0)
    if y0 == y1:
        fig.add_trace(go.Scatter(x=[x0_mmss, x1_mmss], y=[y0, y1], mode='lines', line=dict(color=color, width=2), showlegend=False))
    else:
        arc_start = x1 - (x1 - x0) * 0.2
        arc_start_mmss = seconds_to_mmss(arc_start)
        fig.add_trace(go.Scatter(x=[x0_mmss, arc_start_mmss], y=[y0, y0], mode='lines', line=dict(color=color, width=2), showlegend=False))
        arc_x, arc_y = bezier_arc(arc_start, y0, x1, y1, arc_height=0.7)
        arc_x_mmss = [seconds_to_mmss(x) for x in arc_x]
        fig.add_trace(go.Scatter(x=arc_x_mmss, y=arc_y, mode='lines', line=dict(color=color, width=2), showlegend=False))

fig.update_layout(title='Show Structure (Flat with Quick Arcs, Colored by Level, No Markers)',
                  xaxis_title='Time (MM:SS)',
                  yaxis_title='Level',
                  yaxis=dict(dtick=1),
                  showlegend=False)
fig.show()

/var/folders/pq/yb15b9m50p1fsyjknkf4_xjh0000gn/T/ipykernel_23294/398164338.py:8: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.

